In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt

In [29]:
us_state_to_abbrev = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
    "District of Columbia": "DC",
    "American Samoa": "AS",
    "Guam": "GU",
    "Northern Mariana Islands": "MP",
    "Puerto Rico": "PR",
    "United States Minor Outlying Islands": "UM",
    "Virgin Islands, U.S.": "VI",
} 

abbrev_to_us_state = dict(map(reversed, us_state_to_abbrev.items()))
# https://gist.github.com/rogerallen/1583593

In [3]:
pvi_14 = pd.read_csv('data/pres_results/2008-12_pres_2012-14_dists.csv')
pvi_16 = pd.read_csv('data/pres_results/2008-16_pres_2016_dists.csv')
pvi_18 = pd.read_csv('data/pres_results/2008-16_pres_2018_dists.csv')
pvi_20 = pd.read_csv('data/pres_results/2008-20_pres_2020_dists.csv')
pvi_22 = pd.read_csv('data/pres_results/2020_pres_2022_dists.csv')
pvi_24 = pd.read_csv('data/pres_results/2020-24_pres_2024_dists.csv')
pvi_24.head()

,Calculated by The Downballot,Unnamed: 1,Unnamed: 2,Subscribe to our newsletter,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Follow @the-downballot.com on Bluesky,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15
0,District,Incumbent,Party,2024,NaN,NaN,NaN,NaN,NaN,NaN,2020,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,Harris,Trump,Total,Harris %,Trump %,Margin,NaN,Biden,Trump,Total,Biden %,Trump %,Margin
2,AK-AL,Nick Begich,(R),"140,026","184,458","338,177",41.41%,54.54%,-13.14%,NaN,"153,778","189,951","357,569",43.01%,53.12%,-10.12%
3,AL-01,Barry Moore,(R),"73,003","257,060","332,700",21.94%,77.26%,-55.32%,NaN,"79,112","243,258","325,715",24.29%,74.68%,-50.40%
4,AL-02,Shomari Figures,(D),"155,603","131,721","290,033",53.65%,45.42%,8.23%,NaN,"174,051","135,333","312,225",55.75%,43.34%,12.40%


In [4]:
pd.set_option('display.max_columns', 100)

In [5]:
# Source: Wikipedia, FEC
# Links:
# https://en.wikipedia.org/wiki/2024_United_States_presidential_election#Electoral_results
# https://www.fec.gov/resources/cms-content/documents/federalelections2020.pdf
# https://www.fec.gov/resources/cms-content/documents/federalelections2016.pdf#page=10
# https://www.fec.gov/resources/cms-content/documents/federalelections2012.pdf#page=11
dem_2pv_24 = 75017613 / (75017613 + 77302580) * 100
dem_2pv_20 = 81283501 / (81283501 + 74223975) * 100
dem_2pv_16 = 65853514 / (65853514 + 62984828) * 100
dem_2pv_12 = 65915795 / (65915795 + 60933504) * 100
dem_2pv_08 = 69498516 / (69498516 + 59948323) * 100
dem_2pv_08, dem_2pv_12, dem_2pv_16, dem_2pv_20, dem_2pv_24

(53.68884751214358,
 51.96386225200976,
 51.113288930712876,
 52.26983492420647,
 49.24994613156773)

In [6]:
# 2024 districts wrangling
pvi_24 = pvi_24.iloc[2:]
pvi_24 = pvi_24.set_axis(['district', 'incumbent', 'party', 'dem_24', 'rep_24', 'tot_24', 'dem_pct_24', 'rep_pct_24',
                         'margin_24', 'na1', 'dem_20', 'rep_20', 'tot_20', 'dem_pct_20', 'rep_pct_20', 'margin_20'], axis=1)
pvi_24 = pvi_24.drop(['na1', 'margin_24', 'margin_20'], axis=1)
for col in ['dem_pct_24', 'rep_pct_24', 'dem_pct_20', 'rep_pct_20']:
    pvi_24[col] = pvi_24[col].str.rstrip('%').astype(float)
for col in ['dem_24', 'rep_24', 'tot_24', 'dem_20', 'rep_20', 'tot_20']:
    pvi_24[col] = pvi_24[col].str.replace(',', '').astype(int)
pvi_24['2p_tot_24'] = pvi_24['dem_24'] + pvi_24['rep_24']
pvi_24['dem_2p_24'] = pvi_24['dem_24'] / pvi_24['2p_tot_24'] * 100
pvi_24['rep_2p_24'] = pvi_24['rep_24'] / pvi_24['2p_tot_24'] * 100
pvi_24['2p_tot_20'] = pvi_24['dem_20'] + pvi_24['rep_20']
pvi_24['dem_2p_20'] = pvi_24['dem_20'] / pvi_24['2p_tot_20'] * 100
pvi_24['rep_2p_20'] = pvi_24['rep_20'] / pvi_24['2p_tot_20'] * 100
pvi_24.head()

,district,incumbent,party,dem_24,rep_24,tot_24,dem_pct_24,rep_pct_24,dem_20,rep_20,tot_20,dem_pct_20,rep_pct_20,2p_tot_24,dem_2p_24,rep_2p_24,2p_tot_20,dem_2p_20,rep_2p_20
2,AK-AL,Nick Begich,(R),140026,184458,338177,41.41,54.54,153778,189951,357569,43.01,53.12,324484,43.153437,56.846563,343729,44.738151,55.261849
3,AL-01,Barry Moore,(R),73003,257060,332700,21.94,77.26,79112,243258,325715,24.29,74.68,330063,22.117899,77.882101,322370,24.540745,75.459255
4,AL-02,Shomari Figures,(D),155603,131721,290033,53.65,45.42,174051,135333,312225,55.75,43.34,287324,54.155935,45.844065,309384,56.257273,43.742727
5,AL-03,Mike Rogers,(R),82654,229676,314869,26.25,72.94,93357,225360,322031,28.99,69.98,312330,26.463676,73.536324,318717,29.291503,70.708497
6,AL-04,Robert Aderholt,(R),53098,267953,323449,16.42,82.84,60121,262473,325713,18.46,80.58,321051,16.538805,83.461195,322594,18.636738,81.363262


In [7]:
# Get 2012 presidential results by district for Florida (unavailable in Daily Kos/Downballot dataset)
fl18 = pd.read_csv('data/dra/FL_2018-2020_district_data.csv')

fl18 = fl18[1:]
fl18 = fl18.rename({'E_12_PRES_Dem': 'dem_12', 'E_12_PRES_Rep': 'rep_12', 'E_12_PRES_Total': 'tot_12'}, axis=1)
fl18['district'] = 'FL-' + fl18['Label'].astype(int).map(lambda x: '0' if x < 10 else '') + fl18['Label'].astype(str)
fl18.head(10)

,ID,Label,T_20_CENS_Total,T_20_CENS_White,T_20_CENS_Hispanic,T_20_CENS_Black,T_20_CENS_Asian,T_20_CENS_Native,T_20_CENS_Pacific,V_20_VAP_Total,V_20_VAP_White,V_20_VAP_Hispanic,V_20_VAP_Black,V_20_VAP_Asian,V_20_VAP_Native,V_20_VAP_Pacific,tot_12,dem_12,rep_12,E_16_PRES_Total,E_16_PRES_Dem,E_16_PRES_Rep,E_16-20_COMP_Total,E_16-20_COMP_Dem,E_16-20_COMP_Rep,E_20-24_COMP_Total,E_20-24_COMP_Dem,E_20-24_COMP_Rep,district
1,1,1,807881,565520,61715,119614,35838,26649,3614,636380,462630,41999,84205,26058,20252,2348,349275,106528,242747,0,0,0,361546,106502,245815,376763,107092,265079,FL-01
2,2,2,727856,537827,55005,96863,19323,18882,1680,588566,446456,39333,73084,14061,14572,1101,325069,110727,214342,0,0,0,331097,105030,219713,351029,101211,246787,FL-02
3,3,3,766133,490232,86459,138327,37514,17176,1903,609560,407610,62753,98121,28893,13013,1316,323907,137649,186258,0,0,0,336731,139317,190509,336662,131147,201998,FL-03
4,4,4,871884,612147,86345,98991,57740,17412,2540,691279,502634,61114,71642,41601,13063,1778,371717,123377,248340,0,0,0,416259,146255,261630,451985,158810,288452,FL-04
5,5,5,748910,276549,74491,367467,26769,13892,1824,580527,233591,53084,268194,20547,10706,1286,310953,199964,110989,0,0,0,298765,184042,109283,275219,159089,112996,FL-05
6,6,6,796254,559426,110333,92485,20361,17039,1419,658454,484011,79497,66626,15599,13258,1019,336956,158711,178245,0,0,0,368623,149143,212018,399693,146798,249436,FL-06
7,7,7,788518,421207,207693,104611,53513,15367,2059,634763,356239,156479,77385,40736,11771,1481,329102,164738,164364,0,0,0,349123,181502,159170,341707,172173,165567,FL-07
8,8,8,783753,558427,93322,88802,27720,16848,1983,645163,481232,66755,62431,20758,13057,1404,358897,153020,205877,0,0,0,391293,152531,229797,408402,152188,252114,FL-08
9,9,9,955602,356187,418271,147178,41180,19334,2525,737088,297589,306139,105117,30659,14717,1851,268315,153222,115093,0,0,0,341422,180531,153774,356967,168486,184708,FL-09
10,10,10,873804,291981,262666,248533,57067,14767,2588,669945,241146,193924,178864,43560,11344,1948,268847,165230,103617,0,0,0,305546,185785,113814,299882,169778,126899,FL-10


In [8]:
# 2020 districts wrangling
pvi_20 = pvi_20.iloc[1:]
pvi_20 = pvi_20.set_axis(['district', 'incumbent', 'party', 'na0', 'dem_20', 'rep_20', 'tot_20', 'dem_pct_20', 'rep_pct_20',
                        'na1', 'dem_16', 'rep_16', 'tot_16', 'dem_pct_16', 'rep_pct_16', 'na2', 
                         'dem_12', 'rep_12', 'tot_12', 'dem_pct_12', 'rep_pct_12', 'na3',
                        'dem_08', 'rep_08', 'tot_08', 'dem_pct_08', 'rep_pct_08'], axis=1)
pvi_20 = pvi_20.drop(['na0', 'na1', 'na2', 'na3'], axis=1)
## Florida 2012 nonsense
pvi_20_fl = pvi_20[pvi_20['tot_12'].isna()]
pvi_20_notfl = pvi_20[~pvi_20['tot_12'].isna()]
def get_fl_res_12(district, party):
    # party is 'dem_12', 'rep_12', or 'tot_12'
    df = fl18[fl18['district'] == district]
    return df[party].values[0]
pvi_20_fl['dem_12'] = pvi_20_fl.apply(lambda x: get_fl_res_12(x['district'], 'dem_12'), axis=1)
pvi_20_fl['rep_12'] = pvi_20_fl.apply(lambda x: get_fl_res_12(x['district'], 'rep_12'), axis=1)
pvi_20_fl['tot_12'] = pvi_20_fl.apply(lambda x: get_fl_res_12(x['district'], 'tot_12'), axis=1)
pvi_20 = pd.concat([pvi_20_fl, pvi_20_notfl], axis=0).sort_values(by=['district'])

for yr in ['08']:
    pvi_20 = pvi_20.drop([f'dem_{yr}', f'rep_{yr}', f'tot_{yr}', f'dem_pct_{yr}', f'rep_pct_{yr}'], axis=1)
for col in ['dem_pct_20', 'rep_pct_20', 'dem_pct_16', 'rep_pct_16', 'dem_pct_12', 'rep_pct_12']:
    pvi_20[col] = pvi_20[col].str.rstrip('%').astype(float)
for col in ['dem_20', 'rep_20', 'tot_20', 'dem_16', 'rep_16', 'tot_16', 'dem_12', 'rep_12', 'tot_12']:
    pvi_20[col] = pvi_20[col].astype(str).str.replace(',', '').astype(int)
pvi_20['2p_tot_20'] = pvi_20['dem_20'] + pvi_20['rep_20']
pvi_20['dem_2p_20'] = pvi_20['dem_20'] / pvi_20['2p_tot_20'] * 100
pvi_20['rep_2p_20'] = pvi_20['rep_20'] / pvi_20['2p_tot_20'] * 100
pvi_20['2p_tot_16'] = pvi_20['dem_16'] + pvi_20['rep_16']
pvi_20['dem_2p_16'] = pvi_20['dem_16'] / pvi_20['2p_tot_16'] * 100
pvi_20['rep_2p_16'] = pvi_20['rep_16'] / pvi_20['2p_tot_16'] * 100
pvi_20['2p_tot_12'] = pvi_20['dem_12'] + pvi_20['rep_12']
pvi_20['dem_2p_12'] = pvi_20['dem_12'] / pvi_20['2p_tot_12'] * 100
pvi_20['rep_2p_12'] = pvi_20['rep_12'] / pvi_20['2p_tot_12'] * 100

pvi_20['lean_16'] = pvi_20['dem_2p_16'] - dem_2pv_16
pvi_20['lean_20'] = pvi_20['dem_2p_20'] - dem_2pv_20
pvi_20['lean_12'] = pvi_20['dem_2p_12'] - dem_2pv_12

pvi_20.head()

,district,incumbent,party,dem_20,rep_20,tot_20,dem_pct_20,rep_pct_20,dem_16,rep_16,tot_16,dem_pct_16,rep_pct_16,dem_12,rep_12,tot_12,dem_pct_12,rep_pct_12,2p_tot_20,dem_2p_20,rep_2p_20,2p_tot_16,dem_2p_16,rep_2p_16,2p_tot_12,dem_2p_12,rep_2p_12,lean_16,lean_20,lean_12
1,AK-AL,Mary Peltola,(R),153778,189951,357569,43.0,53.1,116454,163387,309407,37.6,52.8,122640,164676,297625,41.2,55.3,343729,44.738151,55.261849,279841,41.614345,58.385655,287316,42.684710,57.315290,-9.498944,-7.531684,-9.279153
2,AL-01,Jerry Carl,(R),117136,211370,331886,35.3,63.7,103364,192634,303478,34.1,63.5,111735,184786,298837,37.4,61.8,328506,35.657187,64.342813,295998,34.920506,65.079494,296521,37.681985,62.318015,-16.192783,-16.612648,-14.281877
3,AL-02,Barry Moore,(R),107776,195953,306714,35.1,63.9,94299,185505,285664,33.0,64.9,105572,182287,289864,36.4,62.9,303729,35.484264,64.515736,279804,33.701806,66.298194,287859,36.674900,63.325100,-17.411483,-16.785571,-15.288962
4,AL-03,Mike Rogers,(R),109495,212012,324741,33.7,65.3,93300,188477,288776,32.3,65.3,103089,174465,280128,36.8,62.3,321507,34.056801,65.943199,281777,33.111290,66.888710,277554,37.141962,62.858038,-18.001999,-18.213034,-14.821901
5,AL-04,Robert Aderholt,(R),57133,260535,320725,17.8,81.2,50722,233661,290726,17.4,80.4,65818,205423,274505,24.0,74.8,317668,17.985129,82.014871,284383,17.835806,82.164194,271241,24.265506,75.734494,-33.277483,-34.284706,-27.698357


In [9]:
# 2018 districts wrangling
pvi_18 = pvi_18.iloc[1:]
pvi_18 = pvi_18.set_axis(['district', 'incumbent', 'party', 'na0', 'dem_20', 'rep_20', 'tot_20', 'dem_pct_20', 'rep_pct_20',
                        'na1', 'dem_16', 'rep_16', 'tot_16', 'dem_pct_16', 'rep_pct_16', 'na2', 
                         'dem_12', 'rep_12', 'tot_12', 'dem_pct_12', 'rep_pct_12', 'na3',
                        'dem_08', 'rep_08', 'tot_08', 'dem_pct_08', 'rep_pct_08'], axis=1)
pvi_18 = pvi_18.drop(['na0', 'na1', 'na2', 'na3'], axis=1)

## Florida 2012 nonsense
pvi_18_fl = pvi_18[pvi_18['tot_12'].isna()]
pvi_18_notfl = pvi_18[~pvi_18['tot_12'].isna()]
def get_fl_res_12(district, party):
    # party is 'dem_12', 'rep_12', or 'tot_12'
    df = fl18[fl18['district'] == district]
    return df[party].values[0]
pvi_18_fl['dem_12'] = pvi_18_fl.apply(lambda x: get_fl_res_12(x['district'], 'dem_12'), axis=1)
pvi_18_fl['rep_12'] = pvi_18_fl.apply(lambda x: get_fl_res_12(x['district'], 'rep_12'), axis=1)
pvi_18_fl['tot_12'] = pvi_18_fl.apply(lambda x: get_fl_res_12(x['district'], 'tot_12'), axis=1)
pvi_18 = pd.concat([pvi_18_fl, pvi_18_notfl], axis=0).sort_values(by=['district'])

for yr in ['08', '20']:
    pvi_18 = pvi_18.drop([f'dem_{yr}', f'rep_{yr}', f'tot_{yr}', f'dem_pct_{yr}', f'rep_pct_{yr}'], axis=1)
for col in ['dem_pct_12', 'rep_pct_12', 'dem_pct_16', 'rep_pct_16']:
    pvi_18[col] = pvi_18[col].str.rstrip('%').astype(float)
for col in ['dem_12', 'rep_12', 'tot_12', 'dem_16', 'rep_16', 'tot_16']:
    pvi_18[col] = pvi_18[col].astype(str).str.replace(',', '').astype(int)
pvi_18['2p_tot_12'] = pvi_18['dem_12'] + pvi_18['rep_12']
pvi_18['dem_2p_12'] = pvi_18['dem_12'] / pvi_18['2p_tot_12'] * 100
pvi_18['rep_2p_12'] = pvi_18['rep_12'] / pvi_18['2p_tot_12'] * 100
pvi_18['2p_tot_16'] = pvi_18['dem_16'] + pvi_18['rep_16']
pvi_18['dem_2p_16'] = pvi_18['dem_16'] / pvi_18['2p_tot_16'] * 100
pvi_18['rep_2p_16'] = pvi_18['rep_16'] / pvi_18['2p_tot_16'] * 100

pvi_18['lean_16'] = pvi_18['dem_2p_16'] - dem_2pv_16
pvi_18['lean_12'] = pvi_18['dem_2p_12'] - dem_2pv_12

pvi_18.head()

,district,incumbent,party,dem_16,rep_16,tot_16,dem_pct_16,rep_pct_16,dem_12,rep_12,tot_12,dem_pct_12,rep_pct_12,2p_tot_12,dem_2p_12,rep_2p_12,2p_tot_16,dem_2p_16,rep_2p_16,lean_16,lean_12
1,AK-AL,"Young, Don",(R),116454,163387,309407,37.6,52.8,122640,164676,297625,41.2,55.3,287316,42.684710,57.315290,279841,41.614345,58.385655,-9.498944,-9.279153
2,AL-01,"Byrne, Bradley",(R),103364,192634,303478,34.1,63.5,111735,184786,298837,37.4,61.8,296521,37.681985,62.318015,295998,34.920506,65.079494,-16.192783,-14.281877
3,AL-02,"Roby, Martha",(R),94299,185505,285664,33.0,64.9,105572,182287,289864,36.4,62.9,287859,36.674900,63.325100,279804,33.701806,66.298194,-17.411483,-15.288962
4,AL-03,"Rogers, Mike",(R),93300,188477,288776,32.3,65.3,103089,174465,280128,36.8,62.3,277554,37.141962,62.858038,281777,33.111290,66.888710,-18.001999,-14.821901
5,AL-04,"Aderholt, Rob",(R),50722,233661,290726,17.4,80.4,65818,205423,274505,24.0,74.8,271241,24.265506,75.734494,284383,17.835806,82.164194,-33.277483,-27.698357


In [10]:
state_abbrevs = [
    # https://en.wikipedia.org/wiki/List_of_states_and_territories_of_the_United_States#States.
    "AK", "AL", "AR", "AZ", "CA", "CO", "CT", "DE", "FL", "GA", "HI", "IA",
    "ID", "IL", "IN", "KS", "KY", "LA", "MA", "MD", "ME", "MI", "MN", "MO",
    "MS", "MT", "NC", "ND", "NE", "NH", "NJ", "NM", "NV", "NY", "OH", "OK",
    "OR", "PA", "RI", "SC", "SD", "TN", "TX", "UT", "VA", "VT", "WA", "WI",
    "WV", "WY"

    # All hail Jeff Paine for this code snippet: https://gist.github.com/JeffPaine/3083347
]

In [11]:
res_16_dists_22 = pd.DataFrame()

for st in state_abbrevs:
    if st in ['DE', 'AK', 'ND', 'SD', 'WY', 'VT']:
        continue
    df = pd.read_csv(f'data/dra/{st}_2022_district_data.csv')
    df = df[1:].rename({'E_16_PRES_Dem': 'dem_16', 'E_16_PRES_Rep': 'rep_16', 'E_16_PRES_Total': 'tot_16'}, axis=1)
    df['district'] = st + '-' + df['Label'].astype(int).map(lambda x: '0' if x < 10 else '') + df['Label'].astype(str)

    df = df[['district', 'dem_16', 'rep_16', 'tot_16']]

    res_16_dists_22 = pd.concat([res_16_dists_22, df], axis=0)

In [12]:
res_16_dists_22.head()

,district,dem_16,rep_16,tot_16
1,AL-01,101839,189526,301237
2,AL-02,96668,193978,299195
3,AL-03,88876,187604,285575
4,AL-04,53761,235439,298177
5,AL-05,91722,189922,297866


In [13]:
# 2022 districts wrangling
pvi_22 = pvi_22.set_axis(['district', 'incumbent', 'party', 'dem_20', 'rep_20', 'tot_20', 'dem_pct_20', 'rep_pct_20'], axis=1)

def get_16(district, party):
    # party is 'dem_16', 'rep_16', or 'tot_16'
    if district[:2] in ['DE', 'AK', 'ND', 'SD', 'WY', 'VT']:
        df = pvi_20[pvi_20['district'] == district]
        return df[party].values[0]
    
    df = res_16_dists_22[res_16_dists_22['district'] == district]
    return df[party].values[0]

for col in ['dem_16', 'rep_16', 'tot_16']:
    pvi_22[col] = pvi_22.apply(lambda x: get_16(x['district'], col), axis=1)

for col in ['dem_pct_20', 'rep_pct_20']:
    pvi_22[col] = pvi_22[col].str.rstrip('%').astype(float)
for col in ['dem_20', 'rep_20', 'tot_20', 'dem_16', 'rep_16', 'tot_16']:
    pvi_22[col] = pvi_22[col].astype(str).str.replace(',', '').astype(int)

for yr in ['16', '20']:
    pvi_22[f'2p_tot_{yr}'] = pvi_22[f'dem_{yr}'] + pvi_22[f'rep_{yr}']
    pvi_22[f'dem_2p_{yr}'] = pvi_22[f'dem_{yr}'] / pvi_22[f'2p_tot_{yr}'] * 100
    pvi_22[f'rep_2p_{yr}'] = pvi_22[f'rep_{yr}'] / pvi_22[f'2p_tot_{yr}'] * 100

    baseline = dem_2pv_16 if yr == '16' else dem_2pv_20
    pvi_22[f'lean_{yr}'] = pvi_22[f'dem_2p_{yr}'] - baseline
pvi_22.head()

,district,incumbent,party,dem_20,rep_20,tot_20,dem_pct_20,rep_pct_20,dem_16,rep_16,tot_16,2p_tot_16,dem_2p_16,rep_2p_16,lean_16,2p_tot_20,dem_2p_20,rep_2p_20,lean_20
0,AK-AL,Mary Peltola,(D),153778,189951,357569,43.0,53.1,116454,163387,309407,279841,41.614345,58.385655,-9.498944,343729,44.738151,55.261849,-7.531684
1,AL-01,Jerry Carl,(R),115566,208116,327039,35.3,63.6,101839,189526,301237,291365,34.952379,65.047621,-16.160910,323682,35.703561,64.296439,-16.566274
2,AL-02,Barry Moore,(R),111264,205435,319817,34.8,64.2,96668,193978,299195,290646,33.259704,66.740296,-17.853585,316699,35.132413,64.867587,-17.137422
3,AL-03,Mike Rogers,(R),102772,210797,316695,32.5,66.6,88876,187604,285575,276480,32.145544,67.854456,-18.967745,313569,32.774924,67.225076,-19.494911
4,AL-04,Robert Aderholt,(R),60733,262359,326261,18.6,80.4,53761,235439,298177,289200,18.589557,81.410443,-32.523732,323092,18.797432,81.202568,-33.472403


In [14]:
res_16_dists_24 = pd.DataFrame()

for st in state_abbrevs:
    if st in ['DE', 'AK', 'ND', 'SD', 'WY', 'VT']:
        continue
    if st in ['NY', 'GA', 'NC', 'LA', 'AL']:
        df = pd.read_csv(f'data/dra/{st}_2024_district_data.csv')
    else:
        df = pd.read_csv(f'data/dra/{st}_2022_district_data.csv')
    df = df[1:].rename({'E_16_PRES_Dem': 'dem_16', 'E_16_PRES_Rep': 'rep_16', 'E_16_PRES_Total': 'tot_16'}, axis=1)
    df['district'] = st + '-' + df['Label'].astype(int).map(lambda x: '0' if x < 10 else '') + df['Label'].astype(str)

    df = df[['district', 'dem_16', 'rep_16', 'tot_16']]

    res_16_dists_24 = pd.concat([res_16_dists_24, df], axis=0)

In [15]:
res_16_dists_24.head()

,district,dem_16,rep_16,tot_16
1,AL-01,64080,220085,293420
2,AL-02,163388,131758,303364
3,AL-03,79538,200498,289677
4,AL-04,51956,236405,297488
5,AL-05,92178,189897,298086


In [16]:
def get_16(district, party):
    # party is 'dem_16', 'rep_16', or 'tot_16'
    if district[:2] in ['DE', 'AK', 'ND', 'SD', 'WY', 'VT']:
        df = pvi_20[pvi_20['district'] == district]
        return df[party].values[0]
    
    df = res_16_dists_24[res_16_dists_24['district'] == district]
    return df[party].values[0]

for col in ['dem_16', 'rep_16', 'tot_16']:
    pvi_24[col] = pvi_24.apply(lambda x: get_16(x['district'], col), axis=1)

for yr in ['16', '20', '24']:
    pvi_24[f'2p_tot_{yr}'] = pvi_24[f'dem_{yr}'] + pvi_24[f'rep_{yr}']
    pvi_24[f'dem_2p_{yr}'] = pvi_24[f'dem_{yr}'] / pvi_24[f'2p_tot_{yr}'] * 100
    pvi_24[f'rep_2p_{yr}'] = pvi_24[f'rep_{yr}'] / pvi_24[f'2p_tot_{yr}'] * 100

    baseline = dem_2pv_16 if yr == '16' else (dem_2pv_20 if yr == '20' else dem_2pv_24)
    pvi_24[f'lean_{yr}'] = pvi_24[f'dem_2p_{yr}'] - baseline
pvi_24.head()

,district,incumbent,party,dem_24,rep_24,tot_24,dem_pct_24,rep_pct_24,dem_20,rep_20,tot_20,dem_pct_20,rep_pct_20,2p_tot_24,dem_2p_24,rep_2p_24,2p_tot_20,dem_2p_20,rep_2p_20,dem_16,rep_16,tot_16,2p_tot_16,dem_2p_16,rep_2p_16,lean_16,lean_20,lean_24
2,AK-AL,Nick Begich,(R),140026,184458,338177,41.41,54.54,153778,189951,357569,43.01,53.12,324484,43.153437,56.846563,343729,44.738151,55.261849,116454,163387,309407,279841,41.614345,58.385655,-9.498944,-7.531684,-6.096509
3,AL-01,Barry Moore,(R),73003,257060,332700,21.94,77.26,79112,243258,325715,24.29,74.68,330063,22.117899,77.882101,322370,24.540745,75.459255,64080,220085,293420,284165,22.550279,77.449721,-28.563010,-27.729090,-27.132047
4,AL-02,Shomari Figures,(D),155603,131721,290033,53.65,45.42,174051,135333,312225,55.75,43.34,287324,54.155935,45.844065,309384,56.257273,43.742727,163388,131758,303364,295146,55.358365,44.641635,4.245076,3.987438,4.905989
5,AL-03,Mike Rogers,(R),82654,229676,314869,26.25,72.94,93357,225360,322031,28.99,69.98,312330,26.463676,73.536324,318717,29.291503,70.708497,79538,200498,289677,280036,28.402777,71.597223,-22.710512,-22.978332,-22.786270
6,AL-04,Robert Aderholt,(R),53098,267953,323449,16.42,82.84,60121,262473,325713,18.46,80.58,321051,16.538805,83.461195,322594,18.636738,81.363262,51956,236405,297488,288361,18.017693,81.982307,-33.095596,-33.633096,-32.711141


In [17]:
# Get 2008 presidential results by district for Florida (unavailable in Daily Kos/Downballot dataset)
fl16 = pd.read_csv('data/dra/FL_2016_district_data.csv')

In [18]:
fl16 = fl16[1:]
fl16 = fl16.rename({'E_08_PRES_Dem': 'dem_08', 'E_08_PRES_Rep': 'rep_08', 'E_08_PRES_Total': 'tot_08'}, axis=1)
fl16['district'] = 'FL-' + fl16['Label'].astype(int).map(lambda x: '0' if x < 10 else '') + fl16['Label'].astype(str)
fl16.head(10)

,ID,Label,T_10_CENS_Total,T_10_CENS_White,T_10_CENS_Hispanic,T_10_CENS_Black,T_10_CENS_Asian,T_10_CENS_Native,T_10_CENS_Pacific,T_10_CENS_BlackAlone,T_10_CENS_AsianAlone,T_10_CENS_NativeAlone,T_10_CENS_OtherAlone,T_10_CENS_TwoOrMore,V_10_VAP_Total,V_10_VAP_White,V_10_VAP_Hispanic,V_10_VAP_Black,V_10_VAP_Asian,V_10_VAP_Native,V_10_VAP_Pacific,V_10_VAP_BlackAlone,V_10_VAP_AsianAlone,V_10_VAP_NativeAlone,V_10_VAP_OtherAlone,V_10_VAP_TwoOrMore,tot_08,dem_08,rep_08,district
1,1,1,696345,522826,35983,104704,24786,12772,2395,94974,16416,5023,1152,19000,541696,420135,24637,71399,17910,9714,1568,67747,13668,4145,602,9997,358489,114948,240089,FL-01
2,2,2,696345,545446,37802,93188,13413,9392,1065,86229,9934,3399,811,12350,552399,441031,26671,69499,9655,7171,717,66793,7766,2740,406,6696,337842,119522,214752,FL-02
3,3,3,696345,490481,56614,120624,26642,7031,1207,110986,21367,2140,1342,13026,547085,399178,40216,84705,20605,5185,812,80493,17528,1691,749,6921,332479,147210,182117,FL-03
4,4,4,696345,540189,47468,71379,35052,6395,1651,62725,27945,2022,1541,13864,543179,433225,33122,49293,24965,4716,1106,45441,21279,1623,876,7164,359883,124889,232404,FL-04
5,5,5,696345,292916,46744,337652,20462,7064,1318,323710,15939,2278,1229,13183,529910,242631,32539,239030,15272,5260,893,232318,12683,1827,620,6998,321904,204539,115468,FL-05
6,6,6,696345,531853,72538,77413,13678,6709,871,68200,10413,1947,1177,9970,562212,446837,49300,54297,10024,5036,615,50041,8306,1599,703,5228,346168,174083,169142,FL-06
7,7,7,696345,443644,143444,81155,32955,6173,1423,66372,26498,1565,1961,12481,548325,364476,103697,57611,24701,4560,1010,49339,20902,1242,1218,7138,330985,169434,159082,FL-07
8,8,8,696344,538737,62246,75503,18431,6783,1524,65498,13157,2037,1329,12881,559112,449554,42811,50968,13543,4973,1020,46584,10678,1650,782,6693,364396,160138,200862,FL-08
9,9,9,696344,339205,242006,100419,25415,7088,2066,79619,19886,1560,2476,11159,523286,276965,165891,67820,18153,4994,1363,56029,15249,1252,1607,5975,272007,148946,121203,FL-09
10,10,10,696345,284655,168274,206348,42260,6429,2842,186576,35230,1553,4909,14424,521757,231422,119262,141253,31881,4579,1999,129700,27646,1183,3452,8553,267783,163284,103019,FL-10


In [19]:
# Get 2008 presidential results by district for California (unavailable in Daily Kos/Downballot dataset)
ca16 = pd.read_csv('data/dra/CA_2014-18_district_data.csv')

ca16 = ca16[1:]
ca16 = ca16.rename({'E_08_PRES_Dem': 'dem_08', 'E_08_PRES_Rep': 'rep_08', 'E_08_PRES_Total': 'tot_08'}, axis=1)
ca16['district'] = 'CA-' + ca16['Label'].astype(int).map(lambda x: '0' if x < 10 else '') + ca16['Label'].astype(str)
ca16.head(10)

,ID,Label,T_10_CENS_Total,T_10_CENS_White,T_10_CENS_Hispanic,T_10_CENS_Black,T_10_CENS_Asian,T_10_CENS_Native,T_10_CENS_Pacific,T_10_CENS_BlackAlone,T_10_CENS_AsianAlone,T_10_CENS_NativeAlone,T_10_CENS_OtherAlone,T_10_CENS_TwoOrMore,V_10_VAP_Total,V_10_VAP_White,V_10_VAP_Hispanic,V_10_VAP_Black,V_10_VAP_Asian,V_10_VAP_Native,V_10_VAP_Pacific,V_10_VAP_BlackAlone,V_10_VAP_AsianAlone,V_10_VAP_NativeAlone,V_10_VAP_OtherAlone,V_10_VAP_TwoOrMore,tot_08,dem_08,rep_08,district
1,1,1,702905,555872,84261,14388,22959,32280,3061,9066,16472,13249,1232,21656,554136,455291,54690,9560,15333,21950,1994,7446,11975,9738,984,13173,316755,139534,177208,CA-01
2,2,2,702905,511716,116969,16805,34076,31062,3286,11115,23714,14754,2070,21359,555305,423500,78229,12121,24017,20948,2231,9285,18946,10401,1554,12428,346621,255491,91129,CA-02
3,3,3,702906,357493,195247,54256,88532,21214,7913,41303,69665,6112,1875,28113,526206,290101,124246,37234,65491,14119,4950,31967,55605,4599,1521,15749,248827,139674,109154,CA-03
4,4,4,702906,549759,86868,12843,38955,20345,3453,8301,27801,7361,1397,20231,544601,441207,56598,8510,26278,14225,2160,6659,21016,5616,1021,11542,337472,149990,187497,CA-04
5,5,5,702905,370810,180559,56657,89674,17544,7136,45219,73967,4421,1512,23214,544581,313874,117326,39213,67271,12133,4792,33968,59175,3443,1032,13229,297741,215830,81919,CA-05
6,6,6,702905,273251,189445,111524,126639,20733,14843,88343,105178,4236,1763,32171,521275,228050,121466,73353,91097,13926,9984,63267,79465,3361,1187,18339,233674,162932,70736,CA-06
7,7,7,702904,401755,113339,67103,117074,15739,9743,51612,95784,3598,1714,30288,525190,321672,71905,43331,81296,10555,6549,37057,70748,2809,1255,16124,289398,152694,136709,CA-07
8,8,8,702905,352775,248397,66112,29587,21619,5227,52720,19858,6891,1283,18586,503201,281907,152327,40400,20568,14720,3097,35171,15849,5355,854,10068,215140,93613,121523,CA-08
9,9,9,702904,259295,261187,72627,115952,16748,7423,58005,93246,3200,1259,23573,497569,209099,162458,45461,79164,10947,4574,39695,68185,2481,789,12539,215779,125120,90654,CA-09
10,10,10,702905,326037,281702,32370,58306,16975,9007,22840,42338,3747,1423,20768,500233,259025,174471,20220,39477,11377,5943,16501,31362,2991,947,11896,216167,111168,105006,CA-10


In [20]:
# Get 2008 presidential results by district for Virginia (unavailable in Daily Kos/Downballot dataset)
va16 = pd.read_csv('data/dra/VA_2016_district_data.csv')

va16 = va16[1:]
va16 = va16.rename({'E_08_PRES_Dem': 'dem_08', 'E_08_PRES_Rep': 'rep_08', 'E_08_PRES_Total': 'tot_08'}, axis=1)
va16['district'] = 'VA-' + va16['Label'].astype(int).map(lambda x: '0' if x < 10 else '') + va16['Label'].astype(str)
va16.head(10)

,ID,Label,T_10_CENS_Total,T_10_CENS_White,T_10_CENS_Hispanic,T_10_CENS_Black,T_10_CENS_Asian,T_10_CENS_Native,T_10_CENS_Pacific,T_10_CENS_BlackAlone,T_10_CENS_AsianAlone,T_10_CENS_NativeAlone,T_10_CENS_OtherAlone,T_10_CENS_TwoOrMore,V_10_VAP_Total,V_10_VAP_White,V_10_VAP_Hispanic,V_10_VAP_Black,V_10_VAP_Asian,V_10_VAP_Native,V_10_VAP_Pacific,V_10_VAP_BlackAlone,V_10_VAP_AsianAlone,V_10_VAP_NativeAlone,V_10_VAP_OtherAlone,V_10_VAP_TwoOrMore,tot_08,dem_08,rep_08,district
1,1,1,727366,510479,57554,129391,27585,9311,1501,116531,20307,2684,1370,17959,541283,392839,36408,90806,18344,6293,907,85910,15134,2070,597,7969,349739,157072,189997,VA-01
2,2,2,708087,461627,46262,155627,45742,8786,2427,139030,35035,2303,1368,21511,542770,369383,30015,108922,32950,6169,1574,101893,27925,1811,722,10294,326303,159752,163616,VA-02
3,3,3,746645,325346,38460,362075,24745,10286,2156,340518,17252,2632,1185,20445,571848,267674,26434,259998,18139,7275,1402,249447,14239,2122,649,10665,337819,225682,109976,VA-03
4,4,4,727366,361782,32585,314760,17860,7948,1290,301867,13059,2778,1046,13809,563976,293612,22008,233208,13598,5815,843,226828,10943,2240,572,7434,345608,209659,133423,VA-04
5,5,5,727365,531392,22973,156175,13764,5535,592,148287,10996,1543,957,11019,574341,428520,15077,117571,10565,4035,405,114675,8989,1232,499,5191,350173,167885,178907,VA-05
6,6,6,727366,593533,31018,86634,13055,5947,703,78117,10250,1695,921,11618,572702,479309,19899,61280,9405,4360,473,58202,7819,1372,440,5473,327938,137661,186826,VA-06
7,7,7,727366,505183,44656,140137,35774,6669,1073,128768,30533,1762,1581,14557,545932,390585,28826,99870,24706,4555,645,95452,22368,1333,778,6346,360240,164568,192766,VA-07
8,8,8,727366,389808,135594,109721,92997,8391,1916,98638,79447,1471,2298,19539,580212,327441,98819,81916,71695,6139,1394,75614,63975,1170,1331,11416,344461,236141,105502,VA-08
9,9,9,727366,659269,13904,40896,10084,4140,434,36340,8180,1185,500,7881,584877,534086,9226,30631,8169,3263,322,28995,6893,975,248,4356,306771,123425,179006,VA-09
10,10,10,727365,484393,85367,57244,99674,5887,1436,47572,86642,1248,1789,19969,520811,359099,55325,38176,66971,3778,923,34494,61166,870,761,8791,338154,172460,163062,VA-10


In [21]:
pvi_16.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,2016,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,2012,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,2008,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,CD,Incumbent,Party,NaN,Clinton,Trump,Total,Clinton%,Trump%,NaN,Obama,Romney,Total,Obama%,Romney%,NaN,Obama,McCain,Total,Obama%,McCain%
1,AK-AL,"Young, Don",(R),NaN,"116,454","163,387","309,407",37.6%,52.8%,NaN,"122,640","164,676","297,625",41.2%,55.3%,NaN,"123,594","193,841","324,467",38.1%,59.7%
2,AL-01,"Byrne, Bradley",(R),NaN,"103,364","192,634","303,478",34.1%,63.5%,NaN,"111,735","184,786","298,837",37.4%,61.8%,NaN,"115,975","183,753","301,556",38.5%,60.9%
3,AL-02,"Roby, Martha",(R),NaN,"94,299","185,505","285,664",33.0%,64.9%,NaN,"105,572","182,287","289,864",36.4%,62.9%,NaN,"101,076","186,164","288,807",35.0%,64.5%
4,AL-03,"Rogers, Mike",(R),NaN,"93,300","188,477","288,776",32.3%,65.3%,NaN,"103,089","174,465","280,128",36.8%,62.3%,NaN,"103,221","176,434","281,951",36.6%,62.6%


In [22]:
# 2016 districts wrangling
pvi_16 = pvi_16.iloc[1:]
pvi_16 = pvi_16.set_axis(['district', 'incumbent', 'party',
                        'na1', 'dem_16', 'rep_16', 'tot_16', 'dem_pct_16', 'rep_pct_16', 'na2', 
                         'dem_12', 'rep_12', 'tot_12', 'dem_pct_12', 'rep_pct_12', 'na3',
                        'dem_08', 'rep_08', 'tot_08', 'dem_pct_08', 'rep_pct_08'], axis=1)
pvi_16 = pvi_16.drop(['na1', 'na2', 'na3'], axis=1)

## Florida 2012 nonsense
pvi_16_fl = pvi_16[pvi_16['tot_12'].isna()]
pvi_16_notfl = pvi_16[~pvi_16['tot_12'].isna()]
def get_fl_res_12(district, party):
    # party is 'dem_12', 'rep_12', or 'tot_12'
    df = fl18[fl18['district'] == district]
    return df[party].values[0]
pvi_16_fl['dem_12'] = pvi_16_fl.apply(lambda x: get_fl_res_12(x['district'], 'dem_12'), axis=1)
pvi_16_fl['rep_12'] = pvi_16_fl.apply(lambda x: get_fl_res_12(x['district'], 'rep_12'), axis=1)
pvi_16_fl['tot_12'] = pvi_16_fl.apply(lambda x: get_fl_res_12(x['district'], 'tot_12'), axis=1)
pvi_16 = pd.concat([pvi_16_fl, pvi_16_notfl], axis=0).sort_values(by=['district'])

## Florida, California, and Virginia 2008 nonsense
pvi_16_fl = pvi_16[pvi_16['tot_08'].isna()]
pvi_16_notfl = pvi_16[~pvi_16['tot_08'].isna()]
def get_fl_res_08(district, party):
    # party is 'dem_08', 'rep_08', or 'tot_08'
    if district[:2] == 'FL':
        df = fl16[fl16['district'] == district]
    elif district[:2] == 'CA':
        df = ca16[ca16['district'] == district]
    elif district[:2] == 'VA':
        df = va16[va16['district'] == district]
    else:
        raise ValueError('State is not Virginia, California, or Florida')
    return df[party].values[0]
pvi_16_fl['dem_08'] = pvi_16_fl.apply(lambda x: get_fl_res_08(x['district'], 'dem_08'), axis=1)
pvi_16_fl['rep_08'] = pvi_16_fl.apply(lambda x: get_fl_res_08(x['district'], 'rep_08'), axis=1)
pvi_16_fl['tot_08'] = pvi_16_fl.apply(lambda x: get_fl_res_08(x['district'], 'tot_08'), axis=1)
pvi_16 = pd.concat([pvi_16_fl, pvi_16_notfl], axis=0).sort_values(by=['district'])

#for yr in ['16']:
#    pvi_16 = pvi_16.drop([f'dem_{yr}', f'rep_{yr}', f'tot_{yr}', f'dem_pct_{yr}', f'rep_pct_{yr}'], axis=1)
for col in ['dem_pct_12', 'rep_pct_12', 'dem_pct_08', 'rep_pct_08']:
    pvi_16[col] = pvi_16[col].str.rstrip('%').astype(float)
for col in ['dem_12', 'rep_12', 'tot_12', 'dem_08', 'rep_08', 'tot_08', 'dem_16', 'rep_16']:
    pvi_16[col] = pvi_16[col].astype(str).str.replace(',', '').astype(int)
pvi_16['2p_tot_16'] = pvi_16['dem_16'] + pvi_16['rep_16']
pvi_16['dem_2p_16'] = pvi_16['dem_16'] / pvi_16['2p_tot_16'] * 100
pvi_16['rep_2p_16'] = pvi_16['rep_16'] / pvi_16['2p_tot_16'] * 100
pvi_16['2p_tot_12'] = pvi_16['dem_12'] + pvi_16['rep_12']
pvi_16['dem_2p_12'] = pvi_16['dem_12'] / pvi_16['2p_tot_12'] * 100
pvi_16['rep_2p_12'] = pvi_16['rep_12'] / pvi_16['2p_tot_12'] * 100
pvi_16['2p_tot_08'] = pvi_16['dem_08'] + pvi_16['rep_08']
pvi_16['dem_2p_08'] = pvi_16['dem_08'] / pvi_16['2p_tot_08'] * 100
pvi_16['rep_2p_08'] = pvi_16['rep_08'] / pvi_16['2p_tot_08'] * 100

pvi_16['lean_08'] = pvi_16['dem_2p_08'] - dem_2pv_08
pvi_16['lean_12'] = pvi_16['dem_2p_12'] - dem_2pv_12

pvi_16.head()

,district,incumbent,party,dem_16,rep_16,tot_16,dem_pct_16,rep_pct_16,dem_12,rep_12,tot_12,dem_pct_12,rep_pct_12,dem_08,rep_08,tot_08,dem_pct_08,rep_pct_08,2p_tot_16,dem_2p_16,rep_2p_16,2p_tot_12,dem_2p_12,rep_2p_12,2p_tot_08,dem_2p_08,rep_2p_08,lean_08,lean_12
1,AK-AL,"Young, Don",(R),116454,163387,"309,407",37.6%,52.8%,122640,164676,297625,41.2,55.3,123594,193841,324467,38.1,59.7,279841,41.614345,58.385655,287316,42.684710,57.315290,317435,38.935215,61.064785,-14.753632,-9.279153
2,AL-01,"Byrne, Bradley",(R),103364,192634,"303,478",34.1%,63.5%,111735,184786,298837,37.4,61.8,115975,183753,301556,38.5,60.9,295998,34.920506,65.079494,296521,37.681985,62.318015,299728,38.693415,61.306585,-14.995432,-14.281877
3,AL-02,"Roby, Martha",(R),94299,185505,"285,664",33.0%,64.9%,105572,182287,289864,36.4,62.9,101076,186164,288807,35.0,64.5,279804,33.701806,66.298194,287859,36.674900,63.325100,287240,35.188692,64.811308,-18.500155,-15.288962
4,AL-03,"Rogers, Mike",(R),93300,188477,"288,776",32.3%,65.3%,103089,174465,280128,36.8,62.3,103221,176434,281951,36.6,62.6,281777,33.111290,66.888710,277554,37.141962,62.858038,279655,36.910121,63.089879,-16.778726,-14.821901
5,AL-04,"Aderholt, Rob",(R),50722,233661,"290,726",17.4%,80.4%,65818,205423,274505,24.0,74.8,71095,204123,278483,25.5,73.3,284383,17.835806,82.164194,271241,24.265506,75.734494,275218,25.832249,74.167751,-27.856598,-27.698357


In [23]:
# 2014 districts wrangling
pvi_14 = pvi_14.iloc[1:]
pvi_14 = pvi_14.set_axis(['district', 'incumbent', 'party',
                        'na2', 
                         'dem_12', 'rep_12', 'tot_12', 'dem_pct_12', 'rep_pct_12', 'na3',
                        'dem_08', 'rep_08', 'tot_08', 'dem_pct_08', 'rep_pct_08'], axis=1)
pvi_14 = pvi_14.drop(['na2', 'na3'], axis=1)

## California 2008 nonsense
pvi_14_fl = pvi_14[pvi_14['tot_08'].isna()]
pvi_14_notfl = pvi_14[~pvi_14['tot_08'].isna()]
def get_fl_res_08(district, party):
    # party is 'dem_08', 'rep_08', or 'tot_08'
    if district[:2] == 'CA':
        df = ca16[ca16['district'] == district]
    else:
        raise ValueError('State is not California')
    return df[party].values[0]
pvi_14_fl['dem_08'] = pvi_14_fl.apply(lambda x: get_fl_res_08(x['district'], 'dem_08'), axis=1)
pvi_14_fl['rep_08'] = pvi_14_fl.apply(lambda x: get_fl_res_08(x['district'], 'rep_08'), axis=1)
pvi_14_fl['tot_08'] = pvi_14_fl.apply(lambda x: get_fl_res_08(x['district'], 'tot_08'), axis=1)
pvi_14 = pd.concat([pvi_14_fl, pvi_14_notfl], axis=0).sort_values(by=['district'])

for col in ['dem_pct_12', 'rep_pct_12', 'dem_pct_08', 'rep_pct_08']:
    pvi_14[col] = pvi_14[col].str.rstrip('%').astype(float)
for col in ['dem_12', 'rep_12', 'tot_12', 'dem_08', 'rep_08', 'tot_08']:
    pvi_14[col] = pvi_14[col].astype(str).str.replace(',', '').astype(float).astype(int)
pvi_14['2p_tot_12'] = pvi_14['dem_12'] + pvi_14['rep_12']
pvi_14['dem_2p_12'] = pvi_14['dem_12'] / pvi_14['2p_tot_12'] * 100
pvi_14['rep_2p_12'] = pvi_14['rep_12'] / pvi_14['2p_tot_12'] * 100
pvi_14['2p_tot_08'] = pvi_14['dem_08'] + pvi_14['rep_08']
pvi_14['dem_2p_08'] = pvi_14['dem_08'] / pvi_14['2p_tot_08'] * 100
pvi_14['rep_2p_08'] = pvi_14['rep_08'] / pvi_14['2p_tot_08'] * 100

pvi_14['lean_08'] = pvi_14['dem_2p_08'] - dem_2pv_08
pvi_14['lean_12'] = pvi_14['dem_2p_12'] - dem_2pv_12

pvi_14.head()

,district,incumbent,party,dem_12,rep_12,tot_12,dem_pct_12,rep_pct_12,dem_08,rep_08,tot_08,dem_pct_08,rep_pct_08,2p_tot_12,dem_2p_12,rep_2p_12,2p_tot_08,dem_2p_08,rep_2p_08,lean_08,lean_12
1,AK-AL,"Young, Don",(R),122640,164676,297625,41.2,55.3,123594,193841,324467,38.1,59.7,287316,42.684710,57.315290,317435,38.935215,61.064785,-14.753632,-9.279153
2,AL-01,"Byrne, Bradley",(R),111735,184786,298837,37.4,61.8,115975,183753,301556,38.5,60.9,296521,37.681985,62.318015,299728,38.693415,61.306585,-14.995432,-14.281877
3,AL-02,"Roby, Martha",(R),105572,182287,289864,36.4,62.9,101076,186164,288807,35.0,64.5,287859,36.674900,63.325100,287240,35.188692,64.811308,-18.500155,-15.288962
4,AL-03,"Rogers, Mike D.",(R),103089,174465,280128,36.8,62.3,103221,176434,281951,36.6,62.6,277554,37.141962,62.858038,279655,36.910121,63.089879,-16.778726,-14.821901
5,AL-04,"Aderholt, Rob",(R),65818,205423,274505,24.0,74.8,71095,204123,278483,25.5,73.3,271241,24.265506,75.734494,275218,25.832249,74.167751,-27.856598,-27.698357


In [36]:
res_12 = pd.read_csv('transformed/house_res_2012.csv')
res_12 = res_12.drop(['Unnamed: 0'], axis=1)
res_12['state_po'] = res_12['state'].map(lambda x: x.title()).map(us_state_to_abbrev)
res_12['district'] = res_12['state_po'] + '-' + res_12['district'].map(lambda x: 'AL' if x == 0 else (
    f'0{x}' if x < 10 else str(x)
))
res_12 = pd.merge(left=res_12, right=pvi_14[['district', 'dem_2p_12']], on='district', how='left')
res_12.head()

,year,state,state_po,special,district,dem,rep,dem_2p_pct,dem_2p_12
0,2012,ALABAMA,AL,False,AL-01,0.0,196374.0,0.000000,37.681985
1,2012,ALABAMA,AL,False,AL-02,103092.0,180591.0,36.340563,36.674900
2,2012,ALABAMA,AL,False,AL-03,98141.0,175306.0,35.890319,37.141962
3,2012,ALABAMA,AL,False,AL-04,69706.0,199071.0,25.934511,24.265506
4,2012,ALABAMA,AL,False,AL-05,101772.0,189185.0,34.978365,35.302496


In [39]:
pvi_24.to_csv('transformed/pvi/past_pres_results_by24dist.csv')
pvi_22.to_csv('transformed/pvi/past_pres_results_by22dist.csv')
pvi_20.to_csv('transformed/pvi/past_pres_results_by20dist.csv')
pvi_18.to_csv('transformed/pvi/past_pres_results_by18dist.csv')
pvi_16.to_csv('transformed/pvi/past_pres_results_by16dist.csv')
pvi_14.to_csv('transformed/pvi/past_pres_results_by14dist.csv')

res_12.to_csv('transformed/house_and_pres_res_2012.csv')